# HACK AI / Intro to Large Language Modelling
## Prompt InstructABSA with few shot examples

### *Mariam Cook*

### *m.cook6@exeter.ac.uk*

### *University of Exeter Centre for Computational Social Science*


# InstructABSA

In this notebook we locally prompt the fine-tuned InstructABSA model (fine-tuned Tk-Instruct, an instruction tuned model of the T5 LLM for aspect based sentiment analysis) per https://github.com/kevinscaria/instructabsa

A key idea behind InstructABSA is that combining fine-tuning and structured prompting with examples performs better than either alone.

Full credit to Kevin Scaria, Himanshu Gupta, Siddharth Goyal, Saurabh Sawant, Swaroop Mishra, Chitta Baral, 2023
https://aclanthology.org/2024.naacl-short.63/ https://arxiv.org/abs/2302.08624

TK-instruct is a fine-tuned version of T5: https://huggingface.co/allenai/tk-instruct-11b-def

In [ ]:
# record the last date time this notebook was run
from datetime import datetime
now = datetime.now()
current_time = now.strftime("%H:%M:%S")
print("Last run:", now.day, now.strftime("%B %Y"), current_time)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import warnings
warnings.filterwarnings('ignore')

To prevent errors authenticate with Hugging Face. We can safely ignore these warnings, but setting HF_TOKEN will prevent the 429 error from appearing in future runs.


```
from huggingface_hub import login
  login(token="your_token_here")
```



### Aspect term extraction

In the instruction template six examples are given by the authors: 2 positive, 2 negative and 2 neutral

In [ ]:
# load the tokenizer and model

ate_tokenizer = AutoTokenizer.from_pretrained("kevinscaria/ate_tk-instruct-base-def-pos-neg-neut-combined")
ate_model = AutoModelForSeq2SeqLM.from_pretrained("kevinscaria/ate_tk-instruct-base-def-pos-neg-neut-combined")

# set instruction template

bos_instruction = """Definition: The output will be the aspects (both implicit and explicit) which have an associated opinion that are extracted from the input text. In cases where there are no aspects the output should be noaspectterm.
    Positive example 1-
    input: I charge it at night and skip taking the cord with me because of the good battery life.
    output: battery life
    Positive example 2-
    input: I even got my teenage son one, because of the features that it offers, like, iChat, Photobooth, garage band and more!.
    output: features, iChat, Photobooth, garage band
    Negative example 1-
    input: Speaking of the browser, it too has problems.
    output: browser
    Negative example 2-
    input: The keyboard is too slick.
    output: keyboard
    Neutral example 1-
    input: I took it back for an Asus and same thing- blue screen which required me to remove the battery to reset.
    output: battery
    Neutral example 2-
    input: Nightly my computer defrags itself and runs a virus scan.
    output: virus scan
    Now complete the following example-
    input: """
delim_instruct1 = ''
eos_instruct1 = ' \noutput:'

In [ ]:
text = 'The cab ride was amazing but the service was pricey.'
tokenized_text = ate_tokenizer(bos_instruction + text + delim_instruct1 + eos_instruct1, return_tensors="pt")
print('tokenized text', tokenized_text)
output = ate_model.generate(tokenized_text.input_ids)
print('output', output)
print('Model output: ', ate_tokenizer.decode(output[0], skip_special_tokens=True))
result = ate_tokenizer.decode(output[0], skip_special_tokens=True)
print(type(result))
if ',' in result:
  print('splitting result')
  split_result_by_comma = result.split(',')
  print(split_result_by_comma)


In [ ]:
def get_aspect_terms(string_to_assess):
  tokenized_text = ate_tokenizer(bos_instruction + string_to_assess + delim_instruct1 + eos_instruct1, return_tensors="pt")
  output = ate_model.generate(tokenized_text.input_ids)
  result = ate_tokenizer.decode(output[0], skip_special_tokens=True)
  if ',' in result:
    # splitting result by comma
    results_list = result.split(',')
  else:
    results_list = [result]
  return results_list

In [ ]:
some_sentences = ['I am sick of this road with lots of potholes', 'The minimum wage is driving unemployment', 'poverty has gone up because of the delay to universal credit', 'we should increase renewable energy production', 'renewable energy is a waste of time']

for asentence in some_sentences:
  print(get_aspect_terms(asentence))

### Aspect Term Sentiment Classification

In [ ]:
# reload the tokenizer and model before setting the instruction - here I save to a new instance

sentiment_tokenizer = AutoTokenizer.from_pretrained("kevinscaria/atsc_tk-instruct-base-def-pos-neg-neut-combined")
sentiment_model = AutoModelForSeq2SeqLM.from_pretrained("kevinscaria/atsc_tk-instruct-base-def-pos-neg-neut-combined")

bos_instruct = """Definition: The output will be 'positive' if the aspect identified in the sentence contains a positive sentiment. If the sentiment of the identified aspect in the input is negative the answer will be 'negative'.
    Otherwise, the output should be 'neutral'. For aspects which are classified as noaspectterm, the sentiment is none.
    Positive example 1-
    input: With the great variety on the menu , I eat here often and never get bored. The aspect is menu.
    output: positive
    Positive example 2-
    input: Great food, good size menu, great service and an unpretensious setting. The aspect is food.
    output: positive
    Negative example 1-
    input: They did not have mayonnaise, forgot our toast, left out ingredients (ie cheese in an omelet), below hot temperatures and the bacon was so over cooked it crumbled on the plate when you touched it. The aspect is toast.
    output: negative
    Negative example 2-
    input: The seats are uncomfortable if you are sitting against the wall on wooden benches. The aspect is seats.
    output: negative
    Neutral example 1-
    input: I asked for seltzer with lime, no ice. The aspect is seltzer with lime.
    output: neutral
    Neutral example 2-
    input: They wouldnt even let me finish my glass of wine before offering another. The aspect is glass of wine.
    output: neutral
    Now complete the following example-
    input: """
delim_instruct = ' The aspect is '
eos_instruct = '.\noutput:'


In [ ]:
text = 'the cab ride was great but the driver was rude'
print(text)
aspect_term = 'driver'
tokenized_text = sentiment_tokenizer(bos_instruct + text + delim_instruct + aspect_term + eos_instruct, return_tensors="pt")
output = sentiment_model.generate(tokenized_text.input_ids)
print(f'Model output for {aspect_term}: ', sentiment_tokenizer.decode(output[0], skip_special_tokens=True))
aspect_term = 'cab ride'
tokenized_text = sentiment_tokenizer(bos_instruct + text + delim_instruct + aspect_term + eos_instruct, return_tensors="pt")
output = sentiment_model.generate(tokenized_text.input_ids)
print(f'Model output for {aspect_term}: ', sentiment_tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
def get_sentiment(string_to_assess, aspect_to_assess):
  tokenized_text = sentiment_tokenizer(bos_instruct + string_to_assess + delim_instruct + aspect_to_assess + eos_instruct, return_tensors="pt")
  output = sentiment_model.generate(tokenized_text.input_ids)
  myresult = sentiment_tokenizer.decode(output[0], skip_special_tokens=True)
  print(f'Model output for {aspect_to_assess}: ', myresult)
  return myresult

In [ ]:
get_sentiment('the cab ride was great but the driver was rude', 'cab ride')

# Combine ATE and ATSC

In [ ]:
# replace with your own sentences
some_sentences = ['I am sick of this road with lots of potholes', 'The minimum wage is driving unemployment', 'poverty has gone up because of the delay to universal credit', 'we should increase renewable energy production', 'renewable energy is a waste of time']

results = []

for asentence in some_sentences:
  aspect_term_results = get_aspect_terms(asentence)
  for aspect_term in aspect_term_results:
    if aspect_term == 'noaspectterm':
      my_result = (asentence, None, None)
      results.append(my_result)
    else:
      print('checking sentiment classification')
      my_sentiment_classification = get_sentiment(asentence, aspect_term)
      my_result = (asentence, aspect_term, my_sentiment_classification)
      results.append(my_result)


In [ ]:
results

In [ ]:
len(results)

In [ ]:
print(type(results[4][2]))

In [ ]:
for item in results:
  print('SENTENCE: ', item[0], '>>> ASPECT: ', item[1], '>>> SENTIMENT: ', item[2])